In [4]:
!pip install requests beautifulsoup4 pandas html5lib lxml --quiet
!pip install selenium webdriver-manager pandas --quiet


In [6]:
!pip install -U selenium --quiet


In [6]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

options = webdriver.ChromeOptions()
options.add_argument("--disable-gpu")
options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=options)

url = "https://www.moneycontrol.com/markets/indian-indices/top-nsemidsml-400-companies-list/112?classic=true&categoryId=5&ex=N"
driver.get(url)

# wait for first rows
WebDriverWait(driver, 30).until(
    EC.presence_of_element_located((By.XPATH, "//tr[@role='row']"))
)

time.sleep(3)

data_dict = {}  # key = Company Name (for de-duplication)

SCROLL_PAUSE = 2
MAX_SCROLLS = 20   # usually enough for 100–200 rows

for scroll in range(MAX_SCROLLS):
    rows = driver.find_elements(By.XPATH, "//tr[@role='row']")

    for row in rows:
        cols = row.find_elements(By.TAG_NAME, "td")
        if len(cols) >= 9:
            name = cols[0].text.strip()

            if name and name not in data_dict:
                data_dict[name] = [
                    name,
                    cols[1].text.strip(),  # LTP
                    cols[2].text.strip(),  # % Change
                    cols[3].text.strip(),  # Change
                    cols[4].text.strip(),  # Volume
                    cols[5].text.strip(),  # Buy Price
                    cols[6].text.strip(),  # Sell Price
                    cols[7].text.strip(),  # Buy Qty
                    cols[8].text.strip()   # Sell Qty
                ]

    # stop early if we have enough rows
    if len(data_dict) >= 100:
        break

    # scroll down
    driver.execute_script("window.scrollBy(0, 800);")
    time.sleep(SCROLL_PAUSE)

driver.quit()

df = pd.DataFrame(
    list(data_dict.values()),
    columns=[
        "Company Name",
        "LTP",
        "% Change",
        "Change",
        "Volume",
        "Buy Price",
        "Sell Price",
        "Buy Qty",
        "Sell Qty"
    ]
)

print("Rows scraped:", len(df))
display(df.head(10))


Rows scraped: 400


,Company Name,LTP,% Change,Change,Volume,Buy Price,Sell Price,Buy Qty,Sell Qty
0,360 ONE WAM,"1,138.90",-0.38,-4.30,"1,662,158",0.00,"1,138.90",0,"1,923"
1,3M India,"34,875.00",-0.40,-140.00,"3,857","34,875.00",0.00,1,0
2,Aadhar Housing,488.00,1.57,7.55,"251,019",0.00,488.00,0,2
3,Aarti Ind,369.80,1.54,5.60,"417,538",0.00,369.80,0,15
4,AAVAS Financier,"1,458.20",0.17,2.50,"237,823",0.00,"1,458.20",0,88
5,Abbott India,"28,035.00",0.20,55.00,"3,745",0.00,"28,035.00",0,1
6,ACC,"1,751.50",-0.19,-3.40,"220,065","1,751.50",0.00,58,0
7,ACME Solar,232.71,-0.03,-0.07,"1,335,471",0.00,232.71,0,"1,534"
8,Action Const,927.50,0.25,2.30,"121,845",0.00,927.50,0,652
9,Adani Total Gas,568.75,-2.27,-13.20,"760,702",568.75,0.00,505,0


In [ ]:
df.columns = [
    "Company Name",
    "LTP",
    "Change",
    "Change %",
    "Volume",
    "Market Cap"
]


In [7]:
df.to_csv("nse_midsml_400.csv", index=False)
df.to_excel("nse_midsml_400.xlsx", index=False)
